# 01 · Explainer — your first package

Three functions, and it does something real: you put `%%explain` at the top of
a cell, and it tells you what that cell does.

```
get_current_cell()  ──>  ask()  ──>  show_md()
   L0: read              L2: model      L1: write
```

That is the whole shape of every tool in this series. Everything after this
notebook is the same three arrows with more care taken at each one.

This is also the notebook where you build **L2** — the model layer — and where
you meet the constraint that shapes it: there are more of you than there is
server.

In [ ]:
import os, sys

REPO = "https://github.com/xamzar/socratic-watchdog.git"
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

# The finished primitives live one directory up in a checkout. In Colab there
# is no checkout, so fetch one.
if IN_COLAB:
    if not os.path.isdir("socratic-watchdog"):
        get_ipython().system(f"git clone -q {REPO}")
    sys.path.insert(0, "socratic-watchdog/tutorial")
else:
    sys.path.insert(0, os.path.abspath(".."))

# A .env beside the repo is picked up if you have one. Not required.
for env_path in ("../../.env", "socratic-watchdog/.env", ".env"):
    if os.path.isfile(env_path):
        for line in open(env_path):
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.strip().split("=", 1)
                os.environ.setdefault(k, v.strip().strip('"').strip("'"))
        break

# ─────────────────────────────────────────────────────────────────────
#  PUT YOUR ENDPOINTS HERE
#
#  NBKIT_LITELLM_*  qwen3.6-27b   better answers, ~30 students at a time
#  NBKIT_DIVE_*     gemma-4-12b   weaker, everyone at once
#
#  ask() tries LiteLLM first and falls through to DiveAI, so a saturated
#  LiteLLM costs you about a second rather than an error. Include the /v1
#  in the base URL -- only "/chat/completions" is appended for you.
# ─────────────────────────────────────────────────────────────────────
# os.environ["NBKIT_LITELLM_BASE_URL"] = "..."
# os.environ["NBKIT_DIVE_BASE_URL"]    = "..."
# os.environ["NBKIT_DIVE_API_KEY"]     = "..."

if not os.environ.get("NBKIT_DIVE_BASE_URL") and os.environ.get("DEEPSEEK_API_KEY"):
    os.environ["NBKIT_DIVE_BASE_URL"] = "https://api.deepseek.com/v1"
    os.environ["NBKIT_DIVE_MODEL"] = "deepseek-chat"
    os.environ["NBKIT_DIVE_API_KEY"] = os.environ["DEEPSEEK_API_KEY"]

import nbkit
from nbkit import (get_cells, get_cell, get_current_cell, current_index,
                   get_cells_before, format_error, insert_cell_below,
                   replace_current_cell, show_md, show_html, live_display)
from nbkit_ask import ask, stream
import nbkit_hooks
from nbkit_hooks import on_cell_run, on_cell_error, cell_magic, clear_hooks

import nbkit_ask
print("primitives loaded ·", "Colab" if IN_COLAB else "local")
print("endpoints:", [e.name for e in nbkit_ask.endpoints()] or
      "NONE — fill in the block above before running any ask() cell")

In [ ]:
def check(label):
    """Run a test and print one line. Use it as a decorator.

    @check("my function does the thing")
    def _():
        assert my_function(1) == 2

    A failing check prints the reason instead of stopping the notebook, so you
    can run the whole page and see everything that is still red.
    """
    def run(test):
        try:
            test()
            print(f"PASS  {label}")
        except AssertionError as e:
            print(f"FAIL  {label}" + (f"  ->  {e}" if str(e) else ""))
        except Exception as e:
            print(f"ERROR {label}  ->  {type(e).__name__}: {e}")
    return run


print("check() ready")

---
## L2 from scratch

An "OpenAI-compatible chat endpoint" sounds like it needs a library. It does
not. It is a JSON POST to a URL, and the reply is JSON. Roughly forty lines of
standard library, no dependency to install, nothing to keep up to date.

Here is the exact shape of the conversation:

```
POST  {base_url}/chat/completions
      Authorization: Bearer {api_key}

      {"model": "...", "messages": [{"role": "user", "content": "..."}]}

  ->  {"choices": [{"message": {"content": "the answer"}}]}
```

Two details that cost people an afternoon each:

- **`/v1`.** Your code appends `/chat/completions` and nothing else. If the
  server wants `/v1/chat/completions`, the `/v1` has to be part of the base URL
  you configure. It is not added for you.
- **The system message.** Rules and persona go in a separate message with
  `"role": "system"`, not glued onto the front of the user text. Models are
  trained to weigh that role differently, and mixing the two is why an
  instruction like "never give the answer" gets ignored.

In [ ]:
import json, urllib.request

@check("posts to base_url + /chat/completions")
def _():
    seen = fake_post(lambda: ask_once("hello"))
    assert seen.full_url == "http://example/v1/chat/completions", seen.full_url

@check("sends the prompt as a user message")
def _():
    seen = fake_post(lambda: ask_once("hello"))
    assert json.loads(seen.data)["messages"] == [{"role": "user", "content": "hello"}]

@check("sends a system prompt on the system role, first")
def _():
    seen = fake_post(lambda: ask_once("hi", system="Be terse."))
    msgs = json.loads(seen.data)["messages"]
    assert msgs[0] == {"role": "system", "content": "Be terse."}
    assert msgs[1]["role"] == "user"

@check("returns the reply text")
def _():
    assert fake_post(lambda: ask_once("hi"), give="the answer") == "the answer"

@check("sends the bearer token")
def _():
    seen = fake_post(lambda: ask_once("hi"))
    assert seen.headers.get("Authorization") == "Bearer test-key"

That test cell needs a fake server, or every run costs you a real request and a
real wait. `fake_post` swaps `urllib.request.urlopen` for a function that
records what was sent and hands back a canned reply.

Faking the network is not a testing trick you will outgrow. It is how you get a
test suite that runs in a tenth of a second, works on a train, and does not
break when the course server is down.

In [ ]:
import io, contextlib

BASE, KEY, MODEL = "http://example/v1", "test-key", "test-model"

class _Resp(io.BytesIO):
    def __enter__(self): return self
    def __exit__(self, *a): self.close(); return False

def fake_post(run, give="ok"):
    """Run `run()` with urlopen faked. Returns the request, or the reply if used."""
    box = {}
    def urlopen(req, timeout=None):
        box["req"] = req
        return _Resp(json.dumps({"choices": [{"message": {"content": give}}]}).encode())
    real, urllib.request.urlopen = urllib.request.urlopen, urlopen
    try:
        out = run()
    finally:
        urllib.request.urlopen = real
    return out if give != "ok" else box["req"]

print("fake_post ready")

**Give your AI this:**

> Write `ask_once(prompt, system=None, max_tokens=512)` using only the standard
> library. POST to `BASE + "/chat/completions"` with header
> `Authorization: Bearer {KEY}` and a JSON body containing `model` (use
> `MODEL`), `messages`, and `max_tokens`. `messages` is a list: if `system` is
> given, a `{"role": "system", ...}` entry first, then always a
> `{"role": "user", "content": prompt}` entry. Return
> `response["choices"][0]["message"]["content"]`. Use `urllib.request`.

In [ ]:
def ask_once(prompt, system=None, max_tokens=512):
    """Send one prompt to one endpoint and return the reply text."""
    raise NotImplementedError

---
## Why one endpoint is not enough

Your `ask_once` works. Now put it in front of a class.

|  | qwen3.6-27b (LiteLLM) | gemma-4-12b (DiveAI) |
|---|---|---|
| answers | better | adequate |
| at once | about 30 | about 500 |

There are around 500 of you. Thirty get qwen; the rest get a timeout. If your
package treats a timeout as an error, most of the class sees a broken tool
during the one hour they are using it.

So `ask()` takes a **list** of endpoints, tries them in order, and uses the
first that answers — the same shape as `get_cells()` in notebook 00, for the
same reason: no single source works in every situation.

Two consequences, and the second one is the one people miss:

1. A saturated LiteLLM costs you about a second, not an error.
2. **Under load, almost everyone is on gemma.** Which means every prompt you
   write for the rest of this course has to work on the 12B model. If it only
   works on the good one, it does not work.

The finished version is in `../nbkit_ask.py`. Read `_attempt` — it is twenty
lines and it holds the whole policy, so `ask` and `stream` cannot drift apart.

In [ ]:
import nbkit_ask

for e in nbkit_ask.endpoints():
    print(f"{e.name:<9} {e.model:<16} {e.base_url}")

print()
print(ask("In one sentence, what is a Jupyter kernel?"))

---
## The package

Three lines of actual logic. Everything above was the vocabulary.

In [ ]:
EXPLAIN_SYSTEM = (
    "You explain Python to a first-year student. Three sentences at most. "
    "Say what the code does and why someone would write it. "
    "No preamble, no restating the question, no markdown headings."
)

@cell_magic("explain")
def explain(line, cell):
    """%%explain — run the cell, then say what it did."""
    get_ipython().run_cell(cell)
    show_md("---\n" + ask(f"Explain this code:\n\n{cell}", system=EXPLAIN_SYSTEM))

print("%%explain ready — try it on the next cell")

In [ ]:
%%explain
counts = {}
for word in "the quick brown fox jumps over the lazy dog the end".split():
    counts[word] = counts.get(word, 0) + 1
top = max(counts, key=counts.get)
print(top, counts[top])

Look at what `explain` actually is:

```python
get_ipython().run_cell(cell)                    # run the student's code
show_md(ask(f"Explain this code:\n{cell}"))     # L2 into L1
```

The magic decorator handed you `cell` — that is L0, already done for you when
the trigger is a magic. `ask` is L2. `show_md` is L1. Three layers, two lines.

**A thing worth noticing:** the cell magic gets the source *without* the
`%%explain` line, and it does **not** run automatically. You had to call
`run_cell` yourself. That is a feature — notebook 04 inspects code and decides
whether to run it — but it surprises everyone exactly once.

---
## Streaming, so a wait does not look like a crash

On a 12B model a three-sentence answer takes several seconds and a long one
takes half a minute. A cell that sits there with no output reads as frozen, and
students press Interrupt.

`stream()` yields the answer in pieces. Combine it with `live_display()` from
notebook 00 and you get text appearing as it arrives, in one output area that
rewrites itself rather than a hundred stacked lines.

In [ ]:
from IPython.display import Markdown

def explain_streaming(source):
    """Same thing, but the answer appears as it is generated."""
    handle = live_display("<i>thinking…</i>")
    text = ""
    for piece in stream(f"Explain this code:\n\n{source}", system=EXPLAIN_SYSTEM):
        text += piece
        handle.update(Markdown(text))
    return text

_ = explain_streaming("print(sum(int(d) for d in str(2**100)))")

---
## When the model is not there

Run this with the endpoints unreachable and see what happens.

In [ ]:
import os
saved = os.environ.get("NBKIT_DIVE_BASE_URL")
os.environ["NBKIT_DIVE_BASE_URL"] = "http://127.0.0.1:9/v1"
os.environ["NBKIT_LITELLM_BASE_URL"] = "http://127.0.0.1:9/v1"
os.environ["no_proxy"] = "127.0.0.1,localhost"

try:
    ask("anything")
except RuntimeError as e:
    print(e)

os.environ["NBKIT_DIVE_BASE_URL"] = saved
del os.environ["NBKIT_LITELLM_BASE_URL"]

It raises, and the message names every endpoint it tried and what each one
said. That is a deliberate design choice and it is worth arguing about, because
the obvious alternative looks kinder:

```python
except Exception:
    return ""        # don't bother the student
```

Do that and your package appears to work perfectly while saying nothing at all.
A beginner has no way to tell that apart from "the AI had no comment", and they
will spend an hour on it. A traceback that says `Connection refused` is a
better teacher than a polite silence.

The exception to this is anything on an automatic trigger, where a raise would
interrupt unrelated work. That is notebook 03's problem, and it is solved there
by catching at the trigger rather than by hiding the failure here.

---
## Your turn

1. **A tone parameter.** `%%explain brief` should give one sentence,
   `%%explain deep` a paragraph. The magic already receives `line` — that is
   what it is for.
2. **Explain the cell above instead of this one.** You have `get_cells_before(1)`
   from notebook 00. Two lines.
3. **Make it cheap.** Explaining the same cell twice should not call the model
   twice. `functools.lru_cache` on a function that takes the source string is
   the whole answer — and skipping a model call you do not need is the single
   biggest thing you can do for a class of 500.

## Next

Notebook 02 writes back into the notebook, which is where the first genuinely
dangerous mistake becomes possible.